# 05 — Análisis, validación y storytelling

Fase 6 del proyecto. Aquí interpretamos los resultados, respondemos las preguntas clave, construimos la narrativa con los datos, proponemos decisiones basadas en evidencia y discutimos las limitaciones del análisis.

## 1. Interpretación de resultados e insights

Volvemos a las tres preguntas de negocio del [notebook 01, sección 2](01_comprension_problema.ipynb): cuánto invertir, cuándo estrenar y qué tipo de película hacer. Leemos juntas las pruebas del [notebook 03](03_hipotesis_pruebas.ipynb) y los indicadores del [notebook 04](04_kpis.ipynb), separando los hallazgos descriptivos de los resultados respaldados por una prueba estadística.

Esta sección interpreta la evidencia disponible. El desarrollo de la narrativa, las decisiones y la discusión general de limitaciones se construyen a partir de estos hallazgos en la siguiente sección.

### 1.1 Qué población estamos interpretando

La referencia principal son las películas de la **base financiera estrenadas desde 2000**: estrenadas, con presupuesto y recaudación reportados y sin montos menores a 10.000 USD. La vista de toda la base se usa como comparación de sensibilidad al periodo. El año final es el último estreno con información financiera válida, no el año actual.

| Evidencia | Fuente del proyecto | Cómo la usamos |
|---|---|---|
| Población, retorno y umbral de rentabilidad | `peliculas_variables.csv`; notebooks 02 y 03 | Seleccionar las mismas películas y conservar las definiciones |
| Seis KPIs generales y por segmento | `kpis_referencia.csv`; notebook 04, sección 2.7 | Recuperar los indicadores ya calculados y contrastar las proporciones con los datos |
| Significancia, efecto e intervalos | Notebook 03, secciones 2.1, 2.2 y 2.3 | Interpretar las pruebas existentes; no plantear pruebas nuevas |
| Participación de directores | `pelicula_director.csv`; página de géneros y talento | Contar películas únicas de directores con al menos cinco títulos en el periodo |

Los cálculos de esta sección aplican el periodo directamente sobre los CSV y se pueden ejecutar sin Power BI ni variables de otros kernels.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Funciona al ejecutar desde notebooks/ o desde la raíz del proyecto en VS Code.
insights_raiz = Path.cwd() if (Path.cwd() / "data/processed").is_dir() else Path.cwd().parent
insights_ruta = insights_raiz / "data/processed"
insights_peliculas = pd.read_csv(
    insights_ruta / "peliculas_variables.csv",
    parse_dates=["release_date"],
    dtype={"base_financiera": "boolean", "rentable": "boolean", "temporada_alta": "boolean"},
)
insights_referencia = pd.read_csv(insights_ruta / "kpis_referencia.csv")
insights_puente = pd.read_csv(insights_raiz / "data/model/pelicula_director.csv")
insights_anio_corte = 2000
insights_periodo = f"Desde {insights_anio_corte}"
insights_columnas_kpi = [
    "ROI mediano (%)", "% rentables", "% no recuperan presupuesto",
    "Presupuesto mediano (M USD)", "Recaudación total (M USD)", "Multiplicador típico (x)",
]

assert insights_peliculas["id"].is_unique
assert not insights_referencia.duplicated(["vista", "segmento", "periodo"]).any()
insights_finanzas = insights_peliculas.loc[insights_peliculas["base_financiera"].fillna(False)].copy()
insights_recientes = insights_finanzas.loc[insights_finanzas["anio"].ge(insights_anio_corte)].copy()
assert not insights_recientes.empty
assert insights_finanzas[["budget", "revenue", "multiplicador", "roi", "rentable", "temporada_alta"]].notna().all().all()
assert insights_finanzas[["budget", "revenue"]].ge(10_000).all().all()
np.testing.assert_allclose(insights_finanzas["multiplicador"], insights_finanzas["revenue"] / insights_finanzas["budget"])
assert insights_finanzas["rentable"].eq(insights_finanzas["multiplicador"].ge(2)).all()

insights_generales = (insights_referencia.loc[insights_referencia["vista"].eq("General")]
                      .set_index("periodo").loc[["Toda la base", insights_periodo]])
assert insights_generales.loc["Toda la base", "Películas"] == len(insights_finanzas)
assert insights_generales.loc[insights_periodo, "Películas"] == len(insights_recientes)
insights_linea_base = insights_generales.loc[insights_periodo]
print(f"Base financiera: {len(insights_finanzas):,} películas únicas.")
print(f"Referencia principal: {int(insights_recientes['anio'].min())}–{int(insights_recientes['anio'].max())}, "
      f"n = {len(insights_recientes):,}; sin género registrado: {insights_recientes['genero_agrupado'].isna().sum()}.")

Base financiera: 3,204 películas únicas.
Referencia principal: 2000–2016, n = 2,265; sin género registrado: 1.


### 1.2 La referencia cambia al mezclar épocas

Antes de comparar un género o una temporada, ponemos al lado los dos periodos. Los porcentajes están expresados de 0 a 100 y los montos en millones de USD nominales. La tabla conserva los cuatro decimales del archivo de referencia; el texto presenta las cifras aproximadas. El multiplicador típico es la **media geométrica**, mientras que el ROI mediano usa la mediana; son resúmenes distintos.

In [2]:
insights_generales[["Películas", *insights_columnas_kpi]].round(4)

,Películas,ROI mediano (%),% rentables,% no recuperan presupuesto,Presupuesto mediano (M USD),Recaudación total (M USD),Multiplicador típico (x)
periodo,,,,,,,
Toda la base,3204,130.5641,56.211,24.1885,25.515,391472.0840,2.0361
Desde 2000,2265,115.3030,53.245,25.6071,30.000,303605.7488,1.8050


**Lectura.** Para las 2.265 películas de 2000–2016, el ROI mediano es 115,30 %, frente a 130,56 % en toda la base. Los porcentajes de rentables son 53,25 % y 56,21 %, respectivamente. Mezclar las épocas ofrece una referencia de retorno más favorable que la del cine reciente de esta muestra.

Un ROI mediano de 115,30 % significa un multiplicador mediano de aproximadamente 2,15x. El multiplicador típico de 1,81x no lo contradice: la media del logaritmo y la mediana responden de forma distinta a la distribución de retornos. Ninguno de los dos predice el resultado de una película particular.

La diferencia entre periodos muestra sensibilidad a la composición de la muestra. No identifica por sí sola cuánto se debe a inflación, selección de películas antiguas u otros factores. La recaudación total de 303.605,75 M USD desde 2000 representa volumen acumulado de esta base, no ingresos anuales del mercado ni utilidad neta.

### 1.3 No recuperar el presupuesto y no llegar a 2x son situaciones diferentes

Los KPI02 y KPI03 no son complementarios. Entre recaudar una vez el presupuesto y recaudar el doble queda un grupo que supera el coste de producción en taquilla, pero no alcanza el criterio aproximado de rentabilidad del proyecto. Contamos las películas para separar esas tres situaciones.

In [3]:
insights_tramos = pd.Series(np.select(
    [insights_recientes["multiplicador"].lt(1), insights_recientes["multiplicador"].lt(2)],
    ["Taquilla < presupuesto", "Presupuesto ≤ taquilla < 2x"],
    default="Taquilla ≥ 2x presupuesto",
), index=insights_recientes.index)
insights_orden_tramos = ["Taquilla < presupuesto", "Presupuesto ≤ taquilla < 2x", "Taquilla ≥ 2x presupuesto"]
insights_recuperacion = insights_tramos.value_counts().reindex(insights_orden_tramos, fill_value=0).rename("Películas").to_frame()
insights_recuperacion["% de la base 2000–2016"] = insights_recuperacion["Películas"] / len(insights_recientes) * 100
assert insights_recuperacion["Películas"].sum() == len(insights_recientes)
np.testing.assert_allclose(
    insights_recuperacion.iloc[[0, 2]]["% de la base 2000–2016"].to_numpy(),
    insights_linea_base[["% no recuperan presupuesto", "% rentables"]].to_numpy(dtype=float),
    rtol=0, atol=0.000051,  # El CSV de referencia conserva cuatro decimales.
)
insights_recuperacion.round(2)

,Películas,% de la base 2000–2016
Taquilla < presupuesto,580,25.61
Presupuesto ≤ taquilla < 2x,479,21.15
Taquilla ≥ 2x presupuesto,1206,53.25


**Insight.** Desde 2000, el 25,61 % no recuperó el presupuesto en taquilla, el 21,15 % quedó entre 1x y menos de 2x y el 53,25 % llegó al umbral de 2x. Las pequeñas diferencias al sumar porcentajes vienen del redondeo.

El segundo grupo no se puede presentar como utilidad asegurada: la recaudación incluye dinero que no recibe la productora y no están registrados todos los costes. Tampoco llamamos pérdida neta al primer grupo; describimos exactamente lo que se observa: taquilla menor que presupuesto.

### 1.4 Pregunta 1: ¿más presupuesto mejora el retorno por cada dólar?

La prueba de Spearman del notebook 03, sección 2.1, da **ρ = −0,0001 y p = 0,996** en las 2.265 películas desde 2000. No se rechaza H0 con α = 0,05: no encontramos evidencia de una asociación monótona entre presupuesto y multiplicador. Esto no demuestra que el presupuesto sea irrelevante, ni descarta relaciones no monótonas.

Para leer el riesgo observado usamos los mismos cuartiles de presupuesto de esa prueba. Los cortes se calculan sobre el periodo completo y se mantienen fijos para esta comparación.

In [4]:
insights_cortes = insights_recientes["budget"].quantile([0, .25, .5, .75, 1]).to_numpy()
insights_etiquetas = [f"{a / 1e6:g} a {b / 1e6:g}" for a, b in zip(insights_cortes[:-1], insights_cortes[1:])]
insights_cuartiles = pd.cut(insights_recientes["budget"], insights_cortes,
                           labels=insights_etiquetas, include_lowest=True)
insights_presupuesto = (insights_recientes.assign(
    cuartil=insights_cuartiles,
    no_recupera=insights_recientes["multiplicador"].lt(1),
).groupby("cuartil", observed=True).agg(
    **{"Películas": ("id", "nunique"), "% no recuperan presupuesto": ("no_recupera", "mean"),
       "% rentables": ("rentable", "mean")}
))
insights_presupuesto[["% no recuperan presupuesto", "% rentables"]] *= 100
assert insights_presupuesto["Películas"].sum() == len(insights_recientes)
display(insights_presupuesto.rename_axis("Cuartil de presupuesto (M USD), 2000–2016").round(2))
insights_brecha_riesgo = (insights_presupuesto["% no recuperan presupuesto"].iloc[0]
                         - insights_presupuesto["% no recuperan presupuesto"].iloc[-1])
print(f"Brecha descriptiva del riesgo, cuartil más barato menos más caro: {insights_brecha_riesgo:.2f} puntos porcentuales.")

,Películas,% no recuperan presupuesto,% rentables
"Cuartil de presupuesto (M USD), 2000–2016",,,
0.015 a 15,639,31.61,55.09
15 a 30,525,28.19,48.38
30 a 62,536,26.31,48.51
62 a 380,565,15.75,60.18


Brecha descriptiva del riesgo, cuartil más barato menos más caro: 15.86 puntos porcentuales.


**Respuesta e insight.** Más presupuesto no aparece asociado con un aumento sistemático del retorno por dólar en el periodo reciente. Sin embargo, las películas del cuartil de hasta 15 M USD no recuperan su presupuesto en taquilla con mayor frecuencia que las del cuartil superior a 62 M: 31,61 % frente a 15,75 %, una brecha de 15,86 puntos porcentuales.

La segunda comparación es **descriptiva**. Spearman no prueba esas diferencias de proporciones. Por eso no concluimos que aumentar el presupuesto reduzca el riesgo ni que todos los rangos produzcan el mismo retorno. El hallazgo es que retorno relativo y frecuencia de no recuperación cuentan aspectos distintos de la inversión.

La correlación negativa de toda la base (ρ = −0,1368; p = 7,5 × 10⁻¹⁵) no se mantiene al restringir el periodo. Esa sensibilidad impide usar el resultado global como una regla general de que «cuanto más barata, mejor».

### 1.5 Pregunta 2: ¿la temporada alta se asocia con un mejor retorno?

La temporada alta reúne mayo, junio, julio, agosto, noviembre y diciembre, según la definición del notebook 03. La prueba de Welch compara el promedio de `log10(multiplicador)`, que al volver a la escala original se interpreta mediante la razón de medias geométricas.

In [5]:
insights_temporadas = (insights_referencia.loc[
    insights_referencia["vista"].eq("Temporada") & insights_referencia["periodo"].eq(insights_periodo)
].set_index("segmento").loc[["Temporada alta", "Resto del año"]])
assert insights_temporadas["Películas"].sum() == len(insights_recientes)
display(insights_temporadas[["Películas", "Multiplicador típico (x)", "% rentables", "% no recuperan presupuesto"]].round(2))
insights_razon_temporada = (insights_temporadas.loc["Temporada alta", "Multiplicador típico (x)"]
                           / insights_temporadas.loc["Resto del año", "Multiplicador típico (x)"])
print(f"Razón del multiplicador típico, alta / resto: {insights_razon_temporada:.2f}x.")

,Películas,Multiplicador típico (x),% rentables,% no recuperan presupuesto
segmento,,,,
Temporada alta,1092,1.95,57.14,22.16
Resto del año,1173,1.68,49.62,28.82


Razón del multiplicador típico, alta / resto: 1.16x.


**Respuesta e insight.** Desde 2000, la temporada alta tiene un multiplicador típico de 1,95x frente a 1,68x en el resto del año. La razón es **1,16x**, con IC del 95 % de **1,04 a 1,30** y **p = 0,008** en la prueba del notebook 03, sección 2.2. Hay evidencia de diferencia, aunque su magnitud es pequeña frente a la variación entre películas.

Ese 16 % aproximado se refiere al multiplicador típico; no equivale a 16 puntos porcentuales más de películas rentables. Los porcentajes observados de rentables son 57,14 % y 49,62 %, una diferencia descriptiva de unos 7,5 puntos. Welch no contrasta esas proporciones.

La revisión por cuartiles del notebook 03 concentra la evidencia en presupuestos de 30–62 M y superiores a 62 M USD, con razones de 1,28x y 1,24x y valores p de 0,002 y 0,009. En los dos cuartiles inferiores no se encontró evidencia suficiente de diferencia. Esto **no prueba que la temporada solo sirva para películas grandes**: las comparaciones por subgrupos son exploratorias y los intervalos de los grupos baratos permiten efectos en ambas direcciones.

Tampoco se identifica el efecto de cambiar una fecha de estreno. Los grupos pueden diferir en franquicias, distribución o promoción, aspectos que estas pruebas no controlan.

### 1.6 Pregunta 3: ¿qué nos dice el género sobre la rentabilidad?

Usamos el género principal agrupado, igual que la hipótesis 3. Una película de la cohorte reciente no tiene género y queda fuera de esta comparación, pero permanece en la referencia general de 2.265. La prueba y las comparaciones entre géneros trabajan con **2.264 películas**.

In [6]:
insights_generos = (insights_referencia.loc[
    insights_referencia["vista"].eq("Género agrupado") & insights_referencia["periodo"].eq(insights_periodo)
].set_index("segmento").sort_values("% rentables", ascending=False))
assert insights_generos["Películas"].sum() == insights_recientes["genero_agrupado"].notna().sum()
display(insights_generos[["Películas", "ROI mediano (%)", "% rentables", "% no recuperan presupuesto",
                         "Presupuesto mediano (M USD)"]].round(2))
insights_brecha_terror = insights_generos.loc["Horror", "% rentables"] - insights_linea_base["% rentables"]
insights_cuota_accion = (insights_generos.loc["Action", "Recaudación total (M USD)"]
                        / insights_linea_base["Recaudación total (M USD)"] * 100)
print(f"Terror frente a la referencia general reciente: +{insights_brecha_terror:.2f} puntos porcentuales de rentables.")
print(f"Acción: {insights_cuota_accion:.2f} % de la recaudación reciente de la base.")

,Películas,ROI mediano (%),% rentables,% no recuperan presupuesto,Presupuesto mediano (M USD)
segmento,,,,,
Horror,125,249.21,77.60,15.20,12.60
Animation,84,135.30,58.33,19.05,80.00
Adventure,185,123.44,57.30,20.54,85.00
Comedy,464,133.58,56.68,23.92,25.00
Fantasy,64,122.65,56.25,14.06,65.50
Science Fiction,49,157.12,55.10,16.33,75.00
Thriller,102,116.08,54.90,23.53,27.61
Drama,519,92.50,49.33,32.76,20.00
Romance,49,99.25,48.98,22.45,25.00


Terror frente a la referencia general reciente: +24.35 puntos porcentuales de rentables.
Acción: 24.12 % de la recaudación reciente de la base.


**Respuesta e insight.** La chi-cuadrado del notebook 03, sección 2.3, encuentra asociación entre género y rentabilidad: **χ² = 48,76, 11 gl, p = 1,0 × 10⁻⁶ y V de Cramér = 0,147** desde 2000. La asociación es estadísticamente significativa, pero débil.

Terror destaca en la descripción: 125 películas, 77,60 % de rentables, ROI mediano de 249,21 % y presupuesto mediano de 12,6 M USD. Su proporción de rentables está unos 24,35 puntos por encima de la referencia general reciente. Además, explica el 61,0 % del chi-cuadrado de ese periodo. Eso lo convierte en el principal componente del resultado global, no en una garantía para un proyecto nuevo.

La prueba sin terror da p = 0,069. No se rechaza H0 entre los otros géneros; esto no demuestra que sean iguales ni que se haya probado cada diferencia individual. La chi-cuadrado global tampoco es una prueba específica de cada pareja de géneros. La ventaja de terror en los dos cuartiles baratos se estudió después de observar el resultado principal y se interpreta como evidencia exploratoria de apoyo.

**Volumen y retorno se separan.** Acción concentra aproximadamente el 24,12 % de la recaudación de la cohorte reciente, pero tiene 47,57 % de rentables y ROI mediano de 87,90 %, por debajo de la referencia general. Ser un segmento de gran taquilla no implica rendir más por cada dólar invertido. El orden descriptivo de los demás géneros no se presenta como un ranking de diferencias estadísticamente probadas.

### 1.7 Cómo interpretar la vista de talento

La página de géneros y talento muestra directores con al menos cinco películas de la base financiera en el periodo seleccionado. Para leer su total hay que reunir las **películas únicas**, porque una película puede tener más de un director. El umbral se aplica al número de títulos de cada director en 2000–2016; no mide su carrera completa.

In [7]:
assert not insights_puente.duplicated(["pelicula_id", "director_id"]).any()
assert insights_puente["pelicula_id"].isin(insights_peliculas["id"]).all()
insights_creditos_recientes = insights_puente.loc[
    insights_puente["pelicula_id"].isin(insights_recientes["id"]) & insights_puente["director_id"].ne(0)
]
insights_titulos_director = insights_creditos_recientes.groupby("director_id")["pelicula_id"].nunique()
insights_directores_elegibles = insights_titulos_director.loc[insights_titulos_director.ge(5)].index
insights_ids_talento = insights_creditos_recientes.loc[
    insights_creditos_recientes["director_id"].isin(insights_directores_elegibles), "pelicula_id"
].unique()
insights_talento = insights_recientes.loc[insights_recientes["id"].isin(insights_ids_talento)]
assert insights_talento["id"].is_unique
insights_resumen_talento = pd.DataFrame([
    {"Grupo": "Base financiera 2000–2016", "Películas únicas": len(insights_recientes),
     "% rentables": insights_recientes["rentable"].mean() * 100},
    {"Grupo": "Películas de directores con al menos 5 títulos en la cohorte",
     "Películas únicas": len(insights_talento), "% rentables": insights_talento["rentable"].mean() * 100},
]).set_index("Grupo")
print(f"Directores con al menos cinco títulos: {len(insights_directores_elegibles)}.")
insights_resumen_talento.round(2)

Directores con al menos cinco títulos: 101.


,Películas únicas,% rentables
Grupo,,
Base financiera 2000–2016,2265,53.25
Películas de directores con al menos 5 títulos en la cohorte,603,64.18


**Insight descriptivo.** Los 101 directores que cumplen el criterio reúnen 603 películas únicas, con 64,18 % de rentables. El 53,25 % general incluye esas mismas películas: no son dos muestras independientes ni una comparación entre ese grupo y todos los directores restantes.

Este porcentaje no demuestra un efecto del director. Seleccionar personas con varios títulos ya favorece trayectorias observadas con continuidad, y la base no permite separar esa selección de presupuesto, género, distribución u otros factores. No se hizo una prueba de hipótesis sobre talento; por eso este hallazgo se presenta como contexto descriptivo y no como evidencia causal ni como evaluación de contratación.

### 1.8 A quién describen estos hallazgos

La composición por idioma ayuda a acotar la interpretación. Recuperamos los dos grupos que ya calculó el notebook 04, sin comparar por separado idiomas con muy pocos títulos.

In [8]:
insights_idiomas = (insights_referencia.loc[
    insights_referencia["vista"].eq("Idioma") & insights_referencia["periodo"].eq(insights_periodo)
].set_index("segmento"))
assert insights_idiomas["Películas"].sum() == len(insights_recientes)
display(insights_idiomas[["Películas", "% rentables", "% no recuperan presupuesto"]].round(2))
insights_porcentaje_ingles = insights_recientes["original_language"].eq("en").mean() * 100
print(f"Películas en inglés: {insights_porcentaje_ingles:.2f} % de la cohorte reciente.")

,Películas,% rentables,% no recuperan presupuesto
segmento,,,
Inglés (en),2159,53.45,25.24
Otros idiomas,106,49.06,33.02


Películas en inglés: 95.32 % de la cohorte reciente.


**Alcance de la lectura.** El inglés representa 2.159 de 2.265 películas (95,32 %). Las otras 106 combinan distintos idiomas y contextos de producción. Los resultados describen principalmente el cine en inglés con cifras reportadas; no permiten trasladar los porcentajes directamente al mercado colombiano ni atribuir al idioma las diferencias observadas.

La interpretación de épocas sigue el mismo criterio: la década de 2010 está incompleta y la taquilla nominal no permite inferir crecimiento real de la industria. Estas cautelas acotan las afirmaciones de esta sección; la discusión general de limitaciones se desarrolla junto con la narrativa y las decisiones del proyecto.

### 1.9 Hallazgos que conectan las tres preguntas

| Pregunta | Evidencia principal en 2000–2016 | Interpretación que sí sostiene el análisis |
|---|---|---|
| ¿Más presupuesto mejora el retorno por dólar? | H1: n = 2.265; ρ = −0,0001; p = 0,996. KPI03: 31,61 % frente a 15,75 % de no recuperación en los cuartiles extremos | No hay evidencia de una asociación monótona. La frecuencia de no recuperación difiere descriptivamente; retorno relativo y riesgo observado no son equivalentes |
| ¿La temporada alta se asocia con mejor retorno? | H2: n = 1.092 / 1.173; razón de multiplicadores típicos = 1,16x; IC 95 % = 1,04–1,30; p = 0,008 | Hay una asociación de magnitud limitada. Los subgrupos no permiten afirmar que cambiar la fecha causará una mejora ni que la temporada sea irrelevante para producciones pequeñas |
| ¿El género se relaciona con llegar al umbral de 2x? | H3: n = 2.264; p = 1,0 × 10⁻⁶; V = 0,147. KPI02 de terror: 77,60 % en 125 películas | La asociación global es débil y terror explica su mayor parte. Los datos no prueban un ranking completo de géneros ni garantizan la rentabilidad de una película nueva |

Las tres preguntas se complementan: presupuesto, calendario y género describen dimensiones diferentes del resultado. La significancia estadística no reemplaza la magnitud del efecto, el tamaño de muestra ni el criterio de rentabilidad. La siguiente sección puede construir la narrativa y las decisiones a partir de esta evidencia, conservando la distinción entre asociación, descripción y causalidad.